# So sánh thuật toán phát hiện lỗi tư thế làm việc từ landmark MediaPipe Pose

Notebook này dùng dataset `posture_data_2fps.csv`, được tạo từ video tư thế đúng/sai bằng OpenCV và MediaPipe Pose, lấy mẫu 2 frame/giây. Mỗi mẫu gồm 99 đặc trưng landmark: `landmark_0_x`, `landmark_0_y`, `landmark_0_z`, ..., `landmark_32_z`, cùng cột `label`.

- `label = 0`: đúng tư thế.
- `label = 1`: sai tư thế.

Mục tiêu là so sánh nhiều thuật toán phân loại truyền thống và một ANN đơn giản trên cùng tập dữ liệu landmark 99 chiều. Notebook không dùng CNN, YOLO, ảnh gốc, database hoặc GUI.

## Phần 1. Import thư viện

Các thư viện bên dưới phục vụ đọc dữ liệu, chia tập, chuẩn hóa, huấn luyện mô hình, đánh giá và trực quan hóa kết quả. Seed được cố định để kết quả có tính lặp lại tốt hơn giữa các lần chạy.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Output directory:", OUTPUT_DIR)

## Phần 2. Load dataset trên Kaggle

Notebook tự tìm file `posture_data_2fps.csv` trong `/kaggle/input`. Nếu không tìm thấy, notebook thử tìm file dự phòng `posture_data.csv`. Sau khi đọc dữ liệu, ta kiểm tra cấu trúc cơ bản để bảo đảm dataset đúng định dạng: có cột `label`, có đúng 99 đặc trưng landmark và nhãn chỉ gồm 0 hoặc 1.

In [ ]:
def find_dataset_path():
    input_dir = "/kaggle/input"
    primary_name = "posture_data_2fps.csv"
    fallback_name = "posture_data.csv"

    for root, dirs, files in os.walk(input_dir):
        if primary_name in files:
            return os.path.join(root, primary_name)

    for root, dirs, files in os.walk(input_dir):
        if fallback_name in files:
            return os.path.join(root, fallback_name)

    raise FileNotFoundError(
        "Không tìm thấy posture_data_2fps.csv hoặc posture_data.csv trong /kaggle/input. "
        "Hãy kiểm tra dataset đã được Add data vào Kaggle Notebook chưa."
    )


csv_path = find_dataset_path()
df = pd.read_csv(csv_path)

print("Đường dẫn file CSV:", csv_path)
print("Kích thước dữ liệu:", df.shape)
display(df.head())

print("Thông tin DataFrame:")
df.info()

missing_values = int(df.isna().sum().sum())
print("Tổng số missing values:", missing_values)

if "label" not in df.columns:
    raise ValueError("Dataset phải có cột label.")

feature_columns = [col for col in df.columns if col != "label"]
num_features = len(feature_columns)
print("Số feature:", num_features)

if num_features != 99:
    raise ValueError(f"Dataset phải có đúng 99 feature landmark, hiện tại có {num_features} feature.")

unique_labels = sorted(df["label"].dropna().unique().tolist())
print("Các label có trong dataset:", unique_labels)

if not set(unique_labels).issubset({0, 1}):
    raise ValueError("Cột label chỉ được gồm 0 và 1.")

print("Phân bố label trước khi xử lý missing values:")
print(df["label"].value_counts().sort_index())

if missing_values > 0:
    rows_before = len(df)
    df = df.dropna().reset_index(drop=True)
    rows_after = len(df)
    print("Số dòng bị loại do missing values:", rows_before - rows_after)
else:
    print("Không có missing values, không cần loại dòng.")

df["label"] = df["label"].astype(int)

print("Phân bố label sau khi xử lý:")
print(df["label"].value_counts().sort_index())

## Phần 3. Khám phá dữ liệu nhanh

Dataset 2 FPS được chọn làm dataset chính vì nó tăng số mẫu so với 1 FPS, giúp mô hình quan sát nhiều biến thiên tư thế hơn. Đồng thời, 2 FPS vẫn hạn chế trùng lặp quá nhiều giữa các frame liên tiếp so với 5 FPS, giúp giảm nguy cơ dataset chứa quá nhiều mẫu gần giống nhau.

In [ ]:
total_samples = len(df)
total_columns = df.shape[1]
num_features = total_columns - 1
label_counts = df["label"].value_counts().sort_index()
label_percentages = df["label"].value_counts(normalize=True).sort_index() * 100

count_label_0 = int(label_counts.get(0, 0))
count_label_1 = int(label_counts.get(1, 0))
percent_label_0 = float(label_percentages.get(0, 0.0))
percent_label_1 = float(label_percentages.get(1, 0.0))

print("Tổng số mẫu:", total_samples)
print("Số cột:", total_columns)
print("Số đặc trưng:", num_features)
print("Số mẫu label 0 - Correct:", count_label_0)
print("Số mẫu label 1 - Incorrect:", count_label_1)
print(f"Tỷ lệ label 0 - Correct: {percent_label_0:.2f}%")
print(f"Tỷ lệ label 1 - Incorrect: {percent_label_1:.2f}%")

plt.figure(figsize=(7, 5))
bars = plt.bar(["0 - Correct", "1 - Incorrect"], [count_label_0, count_label_1], color=["#2E86AB", "#F18F01"])
plt.title("Phân bố label trong dataset 2 FPS")
plt.xlabel("Label")
plt.ylabel("Số mẫu")
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(int(height)), ha="center", va="bottom")
plt.tight_layout()
label_distribution_path = os.path.join(OUTPUT_DIR, "label_distribution_2fps.png")
plt.savefig(label_distribution_path, dpi=150, bbox_inches="tight")
plt.show()

print("Đã lưu hình:", label_distribution_path)

## Phần 4. Chia train / validation / test

Tất cả thuật toán sử dụng cùng cách chia train, validation và test để đảm bảo so sánh công bằng. Tập test được giữ riêng và chỉ dùng để đánh giá cuối cùng, tránh việc mô hình được hưởng lợi từ một cách chia dữ liệu khác.

- Train: 70%.
- Validation: 15%.
- Test: 15%.

Tham số `stratify` giúp giữ tỷ lệ label 0 và label 1 tương đối ổn định trong từng tập.

In [ ]:
X = df.drop("label", axis=1)
y = df["label"]

X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

val_size = 0.15 / 0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp, y_train_temp,
    test_size=val_size,
    random_state=SEED,
    stratify=y_train_temp
)

print("X_train.shape:", X_train.shape)
print("X_val.shape:", X_val.shape)
print("X_test.shape:", X_test.shape)

def print_label_distribution(name, labels):
    counts = labels.value_counts().sort_index()
    percentages = labels.value_counts(normalize=True).sort_index() * 100
    print(f"\n{name}:")
    for label in [0, 1]:
        count = int(counts.get(label, 0))
        percentage = float(percentages.get(label, 0.0))
        label_name = "Correct" if label == 0 else "Incorrect"
        print(f"  Label {label} - {label_name}: {count} mẫu ({percentage:.2f}%)")

print_label_distribution("Train", y_train)
print_label_distribution("Validation", y_val)
print_label_distribution("Test", y_test)

## Phần 5. Hàm đánh giá chung

Bài toán hướng đến phát hiện sai tư thế, vì vậy class dương được xem là `1 - Incorrect`. Recall của lớp sai tư thế rất quan trọng: recall thấp nghĩa là nhiều trường hợp sai tư thế bị bỏ sót, làm ứng dụng cảnh báo kém hiệu quả.

In [ ]:
def evaluate_predictions(model_name, y_true, y_pred, train_time, predict_time):
    avg_predict_time_ms = (predict_time / len(y_true)) * 1000

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Train Time (s)": train_time,
        "Predict Time (s)": predict_time,
        "Avg Predict Time / Sample (ms)": avg_predict_time_ms
    }


results = []
confusion_matrices = {}
classification_reports = {}
trained_models = {}
test_predictions = {}

## Phần 6. Train các thuật toán scikit-learn

Ta so sánh nhiều nhóm thuật toán để có cái nhìn toàn diện hơn: mô hình tuyến tính, mô hình dựa trên khoảng cách, cây quyết định, ensemble và SVM. Logistic Regression, SVM và KNN dùng `StandardScaler` vì các thuật toán này nhạy với thang đo đặc trưng.

In [ ]:
sklearn_models = [
    (
        "Logistic Regression",
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=SEED))
        ])
    ),
    (
        "SVM",
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", random_state=SEED))
        ])
    ),
    (
        "KNN",
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5))
        ])
    ),
    (
        "Decision Tree",
        DecisionTreeClassifier(random_state=SEED)
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        )
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier(random_state=SEED)
    )
]

for model_name, model in sklearn_models:
    print("=" * 70)
    print("Đang train:", model_name)

    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start_time

    start_time = time.perf_counter()
    y_pred = model.predict(X_test)
    predict_time = time.perf_counter() - start_time

    metrics = evaluate_predictions(model_name, y_test, y_pred, train_time, predict_time)
    results.append(metrics)

    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred, labels=[0, 1])
    classification_reports[model_name] = classification_report(
        y_test,
        y_pred,
        target_names=["0 - Correct", "1 - Incorrect"],
        zero_division=0
    )
    trained_models[model_name] = model
    test_predictions[model_name] = y_pred

    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall: {metrics['Recall']:.4f}")
    print(f"F1-score: {metrics['F1-score']:.4f}")
    print(f"Train Time (s): {metrics['Train Time (s)']:.4f}")
    print(f"Predict Time (s): {metrics['Predict Time (s)']:.4f}")
    print(f"Avg Predict Time / Sample (ms): {metrics['Avg Predict Time / Sample (ms)']:.4f}")

## Phần 7. Train ANN TensorFlow/Keras

ANN được huấn luyện trên dữ liệu landmark đã chuẩn hóa riêng. Validation set được dùng để theo dõi `val_loss`, dừng sớm khi mô hình không còn cải thiện và lưu checkpoint tốt nhất. Mô hình này vẫn chỉ nhận input 99 chiều, không xử lý ảnh gốc.

In [ ]:
scaler_ann = StandardScaler()
X_train_scaled = scaler_ann.fit_transform(X_train)
X_val_scaled = scaler_ann.transform(X_val)
X_test_scaled = scaler_ann.transform(X_test)

ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(99,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()

ann_best_path = os.path.join(OUTPUT_DIR, "ann_best_compare_2fps.keras")
scaler_ann_path = os.path.join(OUTPUT_DIR, "scaler_ann_compare_2fps.pkl")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint(
        ann_best_path,
        save_best_only=True,
        monitor="val_loss",
        mode="min"
    )
]

epochs = 100
batch_size = 32

start_time = time.perf_counter()
history = ann_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1
)
ann_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()
y_prob = ann_model.predict(X_test_scaled)
ann_predict_time = time.perf_counter() - start_time

y_pred_ann = (y_prob >= 0.5).astype(int).ravel()

ann_metrics = evaluate_predictions("ANN", y_test, y_pred_ann, ann_train_time, ann_predict_time)
results.append(ann_metrics)

confusion_matrices["ANN"] = confusion_matrix(y_test, y_pred_ann, labels=[0, 1])
classification_reports["ANN"] = classification_report(
    y_test,
    y_pred_ann,
    target_names=["0 - Correct", "1 - Incorrect"],
    zero_division=0
)
trained_models["ANN"] = ann_model
test_predictions["ANN"] = y_pred_ann

joblib.dump(scaler_ann, scaler_ann_path)

print("=" * 70)
print("Kết quả ANN")
print(f"Accuracy: {ann_metrics['Accuracy']:.4f}")
print(f"Precision: {ann_metrics['Precision']:.4f}")
print(f"Recall: {ann_metrics['Recall']:.4f}")
print(f"F1-score: {ann_metrics['F1-score']:.4f}")
print(f"Train Time (s): {ann_metrics['Train Time (s)']:.4f}")
print(f"Predict Time (s): {ann_metrics['Predict Time (s)']:.4f}")
print(f"Avg Predict Time / Sample (ms): {ann_metrics['Avg Predict Time / Sample (ms)']:.4f}")
print("Đã lưu ANN best model:", ann_best_path)
print("Đã lưu ANN scaler:", scaler_ann_path)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("ANN Training Loss / Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
ann_loss_path = os.path.join(OUTPUT_DIR, "ann_training_loss_2fps.png")
plt.savefig(ann_loss_path, dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("ANN Training Accuracy / Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
ann_accuracy_path = os.path.join(OUTPUT_DIR, "ann_training_accuracy_2fps.png")
plt.savefig(ann_accuracy_path, dpi=150, bbox_inches="tight")
plt.show()

print("Đã lưu hình:", ann_loss_path)
print("Đã lưu hình:", ann_accuracy_path)

## Phần 8. Tạo bảng so sánh

Bảng so sánh được sắp xếp theo F1-score giảm dần. F1-score cân bằng giữa Precision và Recall, phù hợp khi cần đánh giá tổng quan khả năng phát hiện lớp sai tư thế.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

round_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Train Time (s)",
    "Predict Time (s)",
    "Avg Predict Time / Sample (ms)"
]
results_df[round_columns] = results_df[round_columns].round(4)

comparison_path = os.path.join(OUTPUT_DIR, "algorithm_comparison_2fps.csv")
results_df.to_csv(comparison_path, index=False)

display(results_df)
print("Đã lưu bảng so sánh:", comparison_path)

## Phần 9. Vẽ biểu đồ so sánh

Thời gian dự đoán rất quan trọng với ứng dụng realtime qua webcam. Mô hình không chỉ cần chính xác mà còn phải dự đoán đủ nhanh để phản hồi gần thời gian thực khi người dùng đang làm việc trước camera.

In [ ]:
def plot_metric_bar(results_df, metric, title, ylabel, filename):
    plt.figure(figsize=(10, 5))
    bars = plt.bar(results_df["Model"], results_df[metric], color="#3A7CA5")
    plt.title(title)
    plt.xlabel("Thuật toán")
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.grid(axis="y", alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.4f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    output_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Đã lưu hình:", output_path)


plot_metric_bar(
    results_df,
    "Accuracy",
    "So sánh Accuracy theo thuật toán",
    "Accuracy",
    "accuracy_comparison_2fps.png"
)

plot_metric_bar(
    results_df,
    "Precision",
    "So sánh Precision theo thuật toán",
    "Precision",
    "precision_comparison_2fps.png"
)

plot_metric_bar(
    results_df,
    "Recall",
    "So sánh Recall lớp sai tư thế theo thuật toán",
    "Recall",
    "recall_comparison_2fps.png"
)

plot_metric_bar(
    results_df,
    "F1-score",
    "So sánh F1-score theo thuật toán",
    "F1-score",
    "f1_comparison_2fps.png"
)

plot_metric_bar(
    results_df,
    "Avg Predict Time / Sample (ms)",
    "So sánh thời gian dự đoán trung bình trên mỗi mẫu",
    "Avg Predict Time / Sample (ms)",
    "predict_time_comparison_2fps.png"
)

plot_metric_bar(
    results_df,
    "Train Time (s)",
    "So sánh thời gian train theo thuật toán",
    "Train Time (s)",
    "train_time_comparison_2fps.png"
)

## Phần 10. Confusion Matrix

Confusion matrix giúp nhìn rõ các loại lỗi:

- TN: dự đoán đúng tư thế đúng.
- FP: báo sai tư thế khi thực tế đúng.
- FN: bỏ sót sai tư thế.
- TP: phát hiện đúng sai tư thế.

Trong ứng dụng cảnh báo tư thế, FN là loại lỗi cần đặc biệt chú ý vì người dùng sai tư thế nhưng hệ thống không cảnh báo.

In [ ]:
def plot_confusion_matrix(cm, model_name, filename):
    plt.figure(figsize=(5.5, 4.8))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.colorbar()

    tick_marks = np.arange(2)
    labels = ["0 - Correct", "1 - Incorrect"]
    plt.xticks(tick_marks, labels, rotation=30, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    threshold = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > threshold else "black"
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=12)

    plt.tight_layout()
    output_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Đã lưu hình:", output_path)


confusion_plot_targets = {
    "ANN": "confusion_matrix_ann_2fps.png",
    "Random Forest": "confusion_matrix_random_forest_2fps.png",
    "SVM": "confusion_matrix_svm_2fps.png",
    "Logistic Regression": "confusion_matrix_logistic_regression_2fps.png"
}

for model_name, filename in confusion_plot_targets.items():
    if model_name in confusion_matrices:
        plot_confusion_matrix(confusion_matrices[model_name], model_name, filename)

best_model_name = results_df.iloc[0]["Model"]
if best_model_name not in confusion_plot_targets:
    plot_confusion_matrix(
        confusion_matrices[best_model_name],
        f"Best Model - {best_model_name}",
        "confusion_matrix_best_model_2fps.png"
    )
else:
    print("Model tốt nhất đã có confusion matrix riêng:", best_model_name)

## Phần 11. Classification report

Classification report cung cấp Precision, Recall và F1-score theo từng lớp. Report này được in ra notebook và lưu thành file `.txt` để đưa vào phụ lục hoặc phần thực nghiệm của luận văn.

In [ ]:
classification_reports_path = os.path.join(OUTPUT_DIR, "classification_reports_2fps.txt")

with open(classification_reports_path, "w", encoding="utf-8") as f:
    for model_name in results_df["Model"]:
        report = classification_reports[model_name]
        print("=" * 70)
        print(model_name)
        print(report)

        f.write("=" * 70 + "\n")
        f.write(model_name + "\n")
        f.write(report + "\n")

print("Đã lưu classification reports:", classification_reports_path)

## Phần 12. Lưu model tốt nhất

Model tốt nhất được chọn theo F1-score. Nếu model tốt nhất là ANN thì file `.keras` và scaler của ANN đã được lưu ở phần trước. Nếu model tốt nhất là mô hình scikit-learn, notebook lưu object model bằng `joblib` để có thể tái sử dụng trong pipeline dự đoán.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_row = results_df.iloc[0]

print("Thuật toán tốt nhất theo F1-score:", best_model_name)
print(f"Accuracy: {best_row['Accuracy']:.4f}")
print(f"Precision: {best_row['Precision']:.4f}")
print(f"Recall: {best_row['Recall']:.4f}")
print(f"F1-score: {best_row['F1-score']:.4f}")
print(f"Avg Predict Time / Sample (ms): {best_row['Avg Predict Time / Sample (ms)']:.4f}")

best_ml_model_path = os.path.join(OUTPUT_DIR, "best_ml_model_2fps.pkl")

if best_model_name == "ANN":
    print("Best model là ANN. Model ANN đã được lưu tại:", ann_best_path)
    print("Scaler ANN đã được lưu tại:", scaler_ann_path)
else:
    joblib.dump(trained_models[best_model_name], best_ml_model_path)
    print("Đã lưu best sklearn model:", best_ml_model_path)

best_model_info_path = os.path.join(OUTPUT_DIR, "best_model_info_2fps.txt")
with open(best_model_info_path, "w", encoding="utf-8") as f:
    f.write("Best model theo F1-score\n")
    f.write(f"Model: {best_model_name}\n")
    f.write(f"Accuracy: {best_row['Accuracy']:.4f}\n")
    f.write(f"Precision: {best_row['Precision']:.4f}\n")
    f.write(f"Recall: {best_row['Recall']:.4f}\n")
    f.write(f"F1-score: {best_row['F1-score']:.4f}\n")
    f.write(f"Train Time (s): {best_row['Train Time (s)']:.4f}\n")
    f.write(f"Predict Time (s): {best_row['Predict Time (s)']:.4f}\n")
    f.write(f"Avg Predict Time / Sample (ms): {best_row['Avg Predict Time / Sample (ms)']:.4f}\n")

print("Đã lưu thông tin best model:", best_model_info_path)

## Phần 13. Kết luận tự động

Phần này tạo đoạn kết luận tiếng Việt dựa trên kết quả thực nghiệm. Khi viết luận văn, có thể dùng đoạn này làm bản nháp và bổ sung nhận xét học thuật chi tiết hơn.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_row = results_df.iloc[0]
realtime_note = "có khả năng đáp ứng yêu cầu realtime" if best_row["Avg Predict Time / Sample (ms)"] < 33.33 else "cần được tối ưu thêm để đáp ứng realtime ổn định"

conclusion = (
    f"Trên cùng tập kiểm thử của bộ dữ liệu posture_data_2fps.csv, mô hình {best_model_name} "
    f"đạt F1-score cao nhất là {best_row['F1-score']:.4f}, Accuracy là {best_row['Accuracy']:.4f}, "
    f"Recall của lớp sai tư thế là {best_row['Recall']:.4f}. "
    "Vì bài toán hướng đến phát hiện sai tư thế, Recall của lớp incorrect là chỉ số quan trọng, "
    "do hệ thống cần hạn chế bỏ sót các trường hợp người dùng đang ngồi sai tư thế. "
    f"Ngoài ra, thời gian dự đoán trung bình trên mỗi mẫu là {best_row['Avg Predict Time / Sample (ms)']:.4f} ms, "
    f"cho thấy mô hình {realtime_note}."
)

if best_model_name != "ANN":
    conclusion += (
        f" Mặc dù {best_model_name} đạt F1-score cao nhất, ANN vẫn được xem xét do khả năng tích hợp "
        "với pipeline hiện tại và khả năng mở rộng khi bổ sung dữ liệu."
    )

print(conclusion)

with open(best_model_info_path, "a", encoding="utf-8") as f:
    f.write("\nKết luận tự động\n")
    f.write(conclusion + "\n")

## Phần 14. Danh sách file kết quả

Các file dưới đây được lưu trong `/kaggle/working` và có thể tải về từ phần Output của Kaggle Notebook.

In [ ]:
expected_files = [
    "algorithm_comparison_2fps.csv",
    "classification_reports_2fps.txt",
    "best_model_info_2fps.txt",
    "label_distribution_2fps.png",
    "accuracy_comparison_2fps.png",
    "precision_comparison_2fps.png",
    "recall_comparison_2fps.png",
    "f1_comparison_2fps.png",
    "predict_time_comparison_2fps.png",
    "train_time_comparison_2fps.png",
    "ann_training_loss_2fps.png",
    "ann_training_accuracy_2fps.png",
    "confusion_matrix_ann_2fps.png",
    "confusion_matrix_random_forest_2fps.png",
    "confusion_matrix_svm_2fps.png",
    "confusion_matrix_logistic_regression_2fps.png",
    "confusion_matrix_best_model_2fps.png",
    "ann_best_compare_2fps.keras",
    "scaler_ann_compare_2fps.pkl",
    "best_ml_model_2fps.pkl"
]

print("Danh sách file kết quả trong /kaggle/working:")
for filename in expected_files:
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(path):
        print("-", filename)

print("\nCác file confusion_matrix_*.png hiện có:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    if filename.startswith("confusion_matrix_") and filename.endswith(".png"):
        print("-", filename)